In [ ]:
%run ./globalvariables

In [ ]:
dbutils.widgets.dropdown("mode", "before", ["before", "after"])
MODE = dbutils.widgets.get("mode")

In [ ]:
# Compare error count vs run start
count_now = spark.sql(f"SELECT COUNT(*) AS n FROM {INFRA_TABLE}.error_logs WHERE status = 'FAILED'").collect()[0]["n"]

if MODE == "before":
    dbutils.jobs.taskValues.set(key="error_count", value=count_now)
    print(f"error_count before run: {count_now}")
else:
    count_before = dbutils.jobs.taskValues.get(
        taskKey="check_errors_before", key="error_count", default=count_now
    )
    new_errors = count_now - count_before
    print(f"error_count after run: {count_now} (new this run: {new_errors})")
    if new_errors > 0:
        raise Exception(f"{new_errors} new error(s) logged in {INFRA_TABLE}.error_logs this run")